# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one client in one monthly warehouse snapshot. I will use the March 2026 snapshot as the development window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#Feature

*   impressions_90d
*   clicks_90d

*   sessions_90d
*   ctr

*  engagement_rate

#Label

*   is_declining_label

#Context

*   content_id
*   client_id

*   content_age_days
*   month / date field

#Excluded

*   Any field derived from the future outcome or directly from the label, because it would leak the answer into the features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify the data contract with three checks: grain uniqueness, the March 2026 row count and date window, and field availability using IS TRUE.

In [16]:
!pip install -q duckdb huggingface_hub pandas

In [17]:
import duckdb
import getpass

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

token = getpass.getpass("Paste your Hugging Face READ token: ")

con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [token]
)

print("Hugging Face connected successfully!")

Paste your Hugging Face READ token: ··········
Hugging Face connected successfully!


In [18]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

result = con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet('{FACT}/month=2026-02/*.parquet')
    """
)

print(result)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      7355108 │
└──────────────┘



In [19]:
# Query 1 — Grain
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT report_date) AS report_dates
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
""")

┌─────────┬─────────┬───────────────┬──────────────┐
│  rows   │ clients │ content_items │ report_dates │
│  int64  │  int64  │     int64     │    int64     │
├─────────┼─────────┼───────────────┼──────────────┤
│ 7355108 │      54 │        321546 │           28 │
└─────────┴─────────┴───────────────┴──────────────┘

In [20]:
# ML-04 Query 1 — Grain verification for March 2026

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_grain_rows,
    COUNT(*) = COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS grain_is_unique
FROM read_parquet(
    '{FACT}/month=2026-03/*.parquet'
)
""")

┌─────────┬───────────────────┬─────────────────┐
│  rows   │ unique_grain_rows │ grain_is_unique │
│  int64  │       int64       │     boolean     │
├─────────┼───────────────────┼─────────────────┤
│ 9841378 │           9841378 │ true            │
└─────────┴───────────────────┴─────────────────┘

In [21]:
# ML-04 Query 2 — March 2026 row count and date span

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date,
    COUNT(DISTINCT report_date) AS report_dates
FROM read_parquet(
    '{FACT}/month=2026-03/*.parquet'
)
""")

┌─────────┬───────────────────┬──────────────────┬──────────────┐
│  rows   │ first_report_date │ last_report_date │ report_dates │
│  int64  │       date        │       date       │    int64     │
├─────────┼───────────────────┼──────────────────┼──────────────┤
│ 9841378 │ 2026-03-01        │ 2026-03-31       │           31 │
└─────────┴───────────────────┴──────────────────┴──────────────┘

In [23]:
# ML-04 Query 3 — availability

con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions IS TRUE) AS available_rows,
    COUNT(*) FILTER (WHERE gsc_clicks IS TRUE) AS clicks_available_rows,
    COUNT(*) FILTER (WHERE sessions_organic IS TRUE) AS organic_sessions_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────┬────────────────┬───────────────────────┬─────────────────────────────────┐
│ total_rows │ available_rows │ clicks_available_rows │ organic_sessions_available_rows │
│   int64    │     int64      │         int64         │              int64              │
├────────────┼────────────────┼───────────────────────┼─────────────────────────────────┤
│    9841378 │        3611061 │                417981 │                          212643 │
└────────────┴────────────────┴───────────────────────┴─────────────────────────────────┘

In [24]:
con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [26]:
# ML-04 — Five-feature frame

features = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    sessions_organic,
    scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
LIMIT 1000
""").df()

features.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25,<NA>,<NA>


- `gsc_impressions` — Knowable at the decision moment because it comes from Search Console data already available for the page.
- `gsc_clicks` — Knowable at the decision moment because it comes from Search Console data available before the decision.
- `gsc_sum_position` — Knowable at the decision moment because it summarizes the page's observed Search Console position.
- `sessions_organic` — Knowable at the decision moment because it comes from observed organic Analytics sessions.
- `scroll_events` — Knowable at the decision moment because it is an observed engagement signal available in the warehouse.

### Deliberate leakage experiment

I will intentionally add the target label to the feature set to demonstrate leakage. The score should become unrealistically high because the model is given the answer directly. I will then remove the leaked label and keep the honest feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell me the full historical story for every content page. Some pages have missing Search Console or Analytics values, so availability is uneven across the data. I also cannot use the final June 2026 month to develop label logic because it is the sealed outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.